# 🎬 YouTube Automation Tool

**Two steps only:**
1. Run **Cell 1** — installs everything (~1 min, do this once)
2. Run **Cell 2** — opens the web interface (copy the link, open on your phone)

> 💡 Optional: get a free Pexels API key at **pexels.com/api** for real video backgrounds

In [ ]:
# ── CELL 1: Install (run once) ────────────────────────────────────────────────
print('Installing... please wait ~1 minute')
!pip install edge-tts moviepy Pillow requests imageio imageio-ffmpeg gradio -q
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Ready! Now run Cell 2.')

In [ ]:
# ── CELL 2: Launch web interface ──────────────────────────────────────────────
import asyncio, shutil, requests, gradio as gr
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
import edge_tts

# ── voices ────────────────────────────────────────────────────────────────────
VOICES = {
    "🎙 Narrator Male (US)":    "en-US-ChristopherNeural",
    "🎙 Narrator Female (US)":  "en-US-JennyNeural",
    "🎙 Guy (US)": "en-US-GuyNeural",
    "🎙 Aria (US)": "en-US-AriaNeural",
    "🎙 Ryan (UK)": "en-GB-RyanNeural",
    "🎙 Sonia (UK)": "en-GB-SoniaNeural",
}

# ── TTS ───────────────────────────────────────────────────────────────────────
async def _tts(text, voice, path):
    words = []
    comm = edge_tts.Communicate(text, voice)
    with open(path, 'wb') as f:
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                words.append({
                    'word':     chunk['text'],
                    'start':    chunk['offset']   / 10_000_000,
                    'duration': chunk['duration'] / 10_000_000,
                })
    return words

# ── B-roll ────────────────────────────────────────────────────────────────────
def _pexels_clips(query, key, n=2):
    r = requests.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': key},
                     params={'query': query, 'per_page': n, 'orientation': 'landscape'},
                     timeout=15)
    if r.status_code != 200:
        return []
    urls = []
    for v in r.json().get('videos', []):
        files = v.get('video_files', [])
        hd = [f for f in files if f.get('quality') == 'hd' and f.get('width', 0) >= 1280]
        pick = hd or files
        if pick:
            urls.append(pick[0]['link'])
    return urls

def _dl(url, path):
    try:
        r = requests.get(url, stream=True, timeout=60)
        with open(path, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        return True
    except:
        return False

def get_broll(topic, key, tmp, count=6):
    kw = [w.lower() for w in topic.split() if len(w) > 3][:3]
    clips = []
    for i, k in enumerate(kw):
        for j, url in enumerate(_pexels_clips(k, key)):
            p = tmp / f'b_{i}_{j}.mp4'
            if _dl(url, p):
                clips.append(p)
            if len(clips) >= count:
                return clips
    return clips

# ── Video ─────────────────────────────────────────────────────────────────────
def make_video(broll, audio_path, words, out):
    from moviepy.editor import (
        VideoFileClip, AudioFileClip, ColorClip, TextClip,
        concatenate_videoclips, CompositeVideoClip
    )
    SIZE = (1920, 1080)
    audio = AudioFileClip(str(audio_path))
    total = audio.duration

    bg_clips, cur, idx = [], 0.0, 0
    if broll:
        while cur < total:
            p = broll[idx % len(broll)]
            try:
                c = VideoFileClip(str(p)).without_audio().resize(SIZE)
                rem = total - cur
                if c.duration > rem:
                    c = c.subclip(0, rem)
                bg_clips.append(c.set_start(cur))
                cur += c.duration
            except:
                pass
            idx += 1
            if idx > len(broll) * 3:
                break

    if not bg_clips:
        bg_clips = [ColorClip(SIZE, color=(10, 10, 25), duration=total)]

    bg = concatenate_videoclips(bg_clips, method='compose').set_audio(audio)

    chunks, group = [], []
    for w in words:
        group.append(w)
        if len(group) == 7:
            chunks.append(group)
            group = []
    if group:
        chunks.append(group)

    sub_clips = []
    for g in chunks:
        text  = ' '.join(w['word'] for w in g)
        start = g[0]['start']
        end   = min(g[-1]['start'] + g[-1]['duration'], total)
        dur   = end - start
        if dur <= 0:
            continue
        try:
            tc = (TextClip(text, fontsize=56, color='white', font='DejaVu-Sans-Bold',
                           stroke_color='black', stroke_width=2,
                           method='caption', size=(1600, None))
                  .set_start(start).set_duration(dur)
                  .set_position(('center', 880)))
            sub_clips.append(tc)
        except:
            pass

    final = CompositeVideoClip([bg] + sub_clips, size=SIZE)
    final.write_videofile(str(out), fps=30, codec='libx264',
                          audio_codec='aac', preset='medium',
                          threads=2, logger=None)
    audio.close()

# ── Thumbnail ─────────────────────────────────────────────────────────────────
def make_thumbnail(title, out):
    SIZE = (1280, 720)
    img  = Image.new('RGB', SIZE, (15, 15, 35))
    draw = ImageDraw.Draw(img)
    for y in range(SIZE[1] // 2, SIZE[1]):
        draw.line([(0, y), (SIZE[0], y)], fill=(0, 0, 0))
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 72)
    except:
        font = ImageFont.load_default()
    words = title.upper().split()
    lines, line = [], []
    for w in words:
        line.append(w)
        if draw.textbbox((0, 0), ' '.join(line), font=font)[2] > SIZE[0] - 100 and len(line) > 1:
            line.pop()
            lines.append(' '.join(line))
            line = [w]
    if line:
        lines.append(' '.join(line))
    y = (SIZE[1] - len(lines) * 82) // 2 + 80
    for text in lines:
        bb = draw.textbbox((0, 0), text, font=font)
        x  = (SIZE[0] - (bb[2] - bb[0])) // 2
        draw.text((x + 3, y + 3), text, fill=(0, 0, 0), font=font)
        draw.text((x, y),         text, fill=(255, 220, 50), font=font)
        y += 82
    img.save(str(out), 'JPEG', quality=95)

# ── Main pipeline ─────────────────────────────────────────────────────────────
def generate(topic, script, voice_label, pexels_key, progress=gr.Progress()):
    if not topic.strip():
        raise gr.Error('Topic cannot be empty')
    if not script.strip():
        raise gr.Error('Script cannot be empty')

    voice = VOICES[voice_label]
    safe  = ''.join(c if c.isalnum() else '_' for c in topic)[:35].lower()
    tmp   = Path('/content/tmp'); tmp.mkdir(exist_ok=True)
    out_mp4   = Path(f'/content/{safe}.mp4')
    out_thumb = Path(f'/content/{safe}_thumbnail.jpg')

    progress(0.1, desc='Generating voiceover...')
    words = asyncio.run(_tts(script, voice, tmp / 'audio.mp3'))

    broll = []
    if pexels_key.strip():
        progress(0.3, desc='Fetching B-roll from Pexels...')
        broll = get_broll(topic, pexels_key.strip(), tmp)

    progress(0.5, desc='Assembling video (this takes a few minutes)...')
    make_video(broll, tmp / 'audio.mp3', words, out_mp4)

    progress(0.9, desc='Creating thumbnail...')
    make_thumbnail(topic, out_thumb)

    shutil.rmtree(tmp, ignore_errors=True)
    progress(1.0, desc='Done!')

    return str(out_mp4), str(out_thumb)

# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title='YouTube Automation', theme=gr.themes.Soft()) as app:
    gr.Markdown('# 🎬 YouTube Automation Tool')
    gr.Markdown('Fill in the fields below and press **Generate Video**.')

    with gr.Row():
        with gr.Column(scale=1):
            topic   = gr.Textbox(label='📌 Video Topic',
                                  placeholder='e.g. The Fall of the Roman Empire')
            script  = gr.Textbox(label='📝 Script',
                                  placeholder='Paste your script here...\n\nThe longer the script, the longer the video.',
                                  lines=12)
            voice   = gr.Dropdown(label='🎙 Voice',
                                   choices=list(VOICES.keys()),
                                   value=list(VOICES.keys())[0])
            pexels  = gr.Textbox(label='🎥 Pexels API Key (optional)',
                                  placeholder='Paste your free key from pexels.com/api',
                                  type='password')
            btn     = gr.Button('🚀 Generate Video', variant='primary', size='lg')

        with gr.Column(scale=1):
            out_video = gr.Video(label='📹 Your Video')
            out_thumb = gr.Image(label='🖼 Thumbnail')

    btn.click(
        fn=generate,
        inputs=[topic, script, voice, pexels],
        outputs=[out_video, out_thumb],
    )

app.launch(share=True, debug=False)